# Partie III — RNN, LSTM, GRU et Seq2Seq

**Projet Deep Learning — EMSI Casablanca (2025-2026)**

Notebook autonome pour la modélisation séquentielle et la traduction **anglais → français** (corpus Tatoeba / ManyThings).

**Plan**
1. Prétraitement textuel et vocabulaires
2. Modèles de langage (RNN, LSTM, GRU) + perplexité
3. Effet du gradient clipping
4. Seq2Seq (encodeur/décodeur GRU) + BLEU
5. Décodage greedy vs beam search

In [ ]:
# Dépendances : pip install torch scikit-learn matplotlib seaborn pandas numpy nltk tqdm

import json
import math
import os
import random
import re
import ssl
import time
import urllib.error
import urllib.request
import zipfile
from collections import Counter
from copy import deepcopy
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "text"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures" / "sequence"
TABLES_DIR = PROJECT_ROOT / "results" / "tables" / "sequence"
MODELS_DIR = PROJECT_ROOT / "results" / "saved_models" / "sequence"
LOGS_DIR = PROJECT_ROOT / "results" / "logs" / "sequence"
for folder in [DATA_DIR, FIGURES_DIR, TABLES_DIR, MODELS_DIR, LOGS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

SEED = 42
CONFIG = {
    "max_pairs": 6000,
    "min_freq": 2,
    "max_len": 10,
    "batch_size": 64,
    "embedding_dim": 128,
    "hidden_dim": 256,
    "dropout": 0.1,
    "lr": 1e-3,
    "weight_decay": 0.0,
    "lm_epochs": 10,
    "seq2seq_epochs": 12,
    "patience": 3,
    "teacher_forcing_ratio": 0.6,
    "clip_value": 1.0,
    "no_clip_value": float("inf"),
    "beam_width": 3,
    "max_decode_len": 14,
    "translation_prefixes": [
        "i am", "i m", "he is", "he s", "she is", "she s",
        "you are", "you re", "we are", "we re", "they are", "they re",
        "are you", "is he", "is she", "tom is", "tom s",
        "can you", "can i", "do you",
    ],
}

SPECIAL_TOKENS = ["<pad>", "<sos>", "<eos>", "<unk>"]


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Périphérique : {device}")

## 1. Prétraitement textuel

In [ ]:
def normalize_sentence(text):
    text = text.lower().strip()
    text = re.sub(r"([?.!,;:])", r" \1 ", text)
    text = re.sub(r"[^a-zàâçéèêëîïôûùüÿñæœ?.!,;:\s-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text):
    return normalize_sentence(text).split()


class Vocabulary:
    def __init__(self, min_freq=1):
        self.min_freq = min_freq
        self.stoi = {token: idx for idx, token in enumerate(SPECIAL_TOKENS)}
        self.itos = list(SPECIAL_TOKENS)

    def build(self, tokenized_sentences):
        counts = Counter(token for sentence in tokenized_sentences for token in sentence)
        for token, freq in sorted(counts.items()):
            if freq >= self.min_freq and token not in self.stoi:
                self.stoi[token] = len(self.itos)
                self.itos.append(token)

    def encode(self, tokens, add_special=True):
        ids = [self.stoi.get(token, self.stoi["<unk>"]) for token in tokens]
        if add_special:
            return [self.stoi["<sos>"], *ids, self.stoi["<eos>"]]
        return ids

    def decode(self, ids, skip_special=True):
        tokens = []
        for idx in ids:
            token = self.itos[idx]
            if token == "<eos>":
                break
            if skip_special and token in SPECIAL_TOKENS:
                continue
            tokens.append(token)
        return tokens

    @property
    def pad_idx(self):
        return self.stoi["<pad>"]

    @property
    def sos_idx(self):
        return self.stoi["<sos>"]

    @property
    def eos_idx(self):
        return self.stoi["<eos>"]

    def __len__(self):
        return len(self.itos)


def download_fra_eng_dataset(data_dir):
    data_dir.mkdir(parents=True, exist_ok=True)
    zip_path = data_dir / "fra-eng.zip"
    extracted_path = data_dir / "fra.txt"
    if extracted_path.exists():
        return extracted_path
    url = "https://www.manythings.org/anki/fra-eng.zip"
    request = urllib.request.Request(url, headers={"User-Agent": "curl/8.0.1"})
    try:
        context = ssl.create_default_context()
        with urllib.request.urlopen(request, context=context) as response:
            zip_path.write_bytes(response.read())
    except (ssl.SSLCertVerificationError, urllib.error.URLError):
        context = ssl._create_unverified_context()
        with urllib.request.urlopen(request, context=context) as response:
            zip_path.write_bytes(response.read())
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extract("fra.txt", path=data_dir)
    return extracted_path


def load_translation_pairs(max_pairs, max_len, prefixes):
    file_path = download_fra_eng_dataset(DATA_DIR)
    pairs = []
    prefix_tuple = tuple(prefixes)
    with file_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            parts = line.strip().split("\t")
            if len(parts) < 2:
                continue
            source = normalize_sentence(parts[0])
            target = normalize_sentence(parts[1])
            if prefix_tuple and not source.startswith(prefix_tuple):
                continue
            if 1 < len(source.split()) <= max_len and 1 < len(target.split()) <= max_len:
                pairs.append((source, target))
    random.Random(SEED).shuffle(pairs)
    return pairs[:max_pairs]


pairs = load_translation_pairs(
    CONFIG["max_pairs"], CONFIG["max_len"], CONFIG["translation_prefixes"]
)
print(f"Paires chargées : {len(pairs)}")
print("Exemple :", pairs[0])

In [ ]:
class LanguageModelingDataset(Dataset):
    def __init__(self, sentences):
        self.samples = [
            (torch.tensor(seq[:-1], dtype=torch.long), torch.tensor(seq[1:], dtype=torch.long))
            for seq in sentences
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


class TranslationDataset(Dataset):
    def __init__(self, encoded_pairs):
        self.pairs = [
            (torch.tensor(src, dtype=torch.long), torch.tensor(tgt, dtype=torch.long))
            for src, tgt in encoded_pairs
        ]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        return self.pairs[index]


def lm_collate_fn(batch, pad_idx):
    inputs, targets = zip(*batch)
    return (
        pad_sequence(inputs, batch_first=True, padding_value=pad_idx),
        pad_sequence(targets, batch_first=True, padding_value=pad_idx),
        torch.tensor([len(item) for item in inputs], dtype=torch.long),
    )


def translation_collate_fn(batch, src_pad_idx, tgt_pad_idx):
    src_batch, tgt_batch = zip(*batch)
    return (
        pad_sequence(src_batch, batch_first=True, padding_value=src_pad_idx),
        pad_sequence(tgt_batch, batch_first=True, padding_value=tgt_pad_idx),
        torch.tensor([len(item) for item in src_batch], dtype=torch.long),
        torch.tensor([len(item) for item in tgt_batch], dtype=torch.long),
    )


train_pairs, test_pairs = train_test_split(pairs, test_size=0.1, random_state=SEED)
train_pairs, val_pairs = train_test_split(train_pairs, test_size=0.1111, random_state=SEED)

train_src = [tokenize(src) for src, _ in train_pairs]
train_tgt = [tokenize(tgt) for _, tgt in train_pairs]
src_vocab = Vocabulary(min_freq=CONFIG["min_freq"])
tgt_vocab = Vocabulary(min_freq=CONFIG["min_freq"])
src_vocab.build(train_src)
tgt_vocab.build(train_tgt)


def encode_pairs(raw_pairs):
    encoded = []
    for src, tgt in raw_pairs:
        encoded.append((src_vocab.encode(tokenize(src)), tgt_vocab.encode(tokenize(tgt))))
    return encoded


train_encoded = encode_pairs(train_pairs)
val_encoded = encode_pairs(val_pairs)
test_encoded = encode_pairs(test_pairs)

train_lm_loader = DataLoader(
    LanguageModelingDataset([tgt for _, tgt in train_encoded]),
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    collate_fn=lambda batch: lm_collate_fn(batch, tgt_vocab.pad_idx),
)
val_lm_loader = DataLoader(
    LanguageModelingDataset([tgt for _, tgt in val_encoded]),
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    collate_fn=lambda batch: lm_collate_fn(batch, tgt_vocab.pad_idx),
)
test_lm_loader = DataLoader(
    LanguageModelingDataset([tgt for _, tgt in test_encoded]),
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    collate_fn=lambda batch: lm_collate_fn(batch, tgt_vocab.pad_idx),
)

train_translation_loader = DataLoader(
    TranslationDataset(train_encoded),
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    collate_fn=lambda batch: translation_collate_fn(batch, src_vocab.pad_idx, tgt_vocab.pad_idx),
)
val_translation_loader = DataLoader(
    TranslationDataset(val_encoded),
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    collate_fn=lambda batch: translation_collate_fn(batch, src_vocab.pad_idx, tgt_vocab.pad_idx),
)
test_translation_loader = DataLoader(
    TranslationDataset(test_encoded),
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    collate_fn=lambda batch: translation_collate_fn(batch, src_vocab.pad_idx, tgt_vocab.pad_idx),
)

summary_df = pd.DataFrame(
    [
        {"split": "entraînement", "paires": len(train_pairs), "vocab_source": len(src_vocab), "vocab_cible": len(tgt_vocab)},
        {"split": "validation", "paires": len(val_pairs), "vocab_source": len(src_vocab), "vocab_cible": len(tgt_vocab)},
        {"split": "test", "paires": len(test_pairs), "vocab_source": len(src_vocab), "vocab_cible": len(tgt_vocab)},
    ]
)
summary_df.to_csv(TABLES_DIR / "sequence_dataset_summary.csv", index=False)
summary_df.to_latex(TABLES_DIR / "sequence_dataset_summary.tex", index=False, float_format="%.4f")
display(summary_df)

## 2. Modèles récurrents et utilitaires

In [ ]:
def perplexity_from_loss(loss_value):
    return float(math.exp(loss_value))


class RecurrentLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, cell_type="rnn", dropout=0.2, pad_idx=0):
        super().__init__()
        self.cell_type = cell_type
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        if cell_type == "rnn":
            self.recurrent = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        elif cell_type == "lstm":
            self.recurrent = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        elif cell_type == "gru":
            self.recurrent = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        else:
            raise ValueError(f"Type inconnu : {cell_type}")
        self.dropout = nn.Dropout(dropout)
        self.output_layer = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)
        outputs, hidden = self.recurrent(embeddings)
        outputs = self.dropout(outputs)
        return self.output_layer(outputs), hidden


class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, pad_idx, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.dropout = nn.Dropout(dropout)
        self.rnn = nn.GRU(embedding_dim, hidden_dim, batch_first=True)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        return self.rnn(embedded)


class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, pad_idx, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.dropout = nn.Dropout(dropout)
        self.rnn = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        self.output_layer = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_token, hidden):
        embedded = self.dropout(self.embedding(input_token.unsqueeze(1)))
        output, hidden = self.rnn(embedded, hidden)
        return self.output_layer(output.squeeze(1)), hidden


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, tgt_pad_idx):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.tgt_pad_idx = tgt_pad_idx

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size, tgt_len = tgt.shape
        vocab_size = self.decoder.output_layer.out_features
        outputs = torch.zeros(batch_size, tgt_len - 1, vocab_size, device=src.device)
        _, hidden = self.encoder(src)
        input_token = tgt[:, 0]
        for timestep in range(1, tgt_len):
            logits, hidden = self.decoder(input_token, hidden)
            outputs[:, timestep - 1, :] = logits
            teacher = random.random() < teacher_forcing_ratio
            input_token = tgt[:, timestep] if teacher else logits.argmax(dim=1)
        return outputs

## 3. Entraînement des modèles de langage (RNN / LSTM / GRU)

In [ ]:
def plot_training_curves(history, title, right_label="Perplexité"):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history["train_loss"], label="Train")
    axes[0].plot(history["val_loss"], label="Val")
    axes[0].set_title(f"{title} — pertes")
    axes[0].set_xlabel("Époque")
    axes[0].legend()
    axes[1].plot(history["train_accuracy"], label=f"{right_label} train")
    axes[1].plot(history["val_accuracy"], label=f"{right_label} val")
    axes[1].set_title(f"{title} — {right_label.lower()}")
    axes[1].set_xlabel("Époque")
    axes[1].legend()
    plt.tight_layout()
    return fig


def lm_train_epoch(model, loader, optimizer, criterion, clip_value):
    model.train()
    total_loss = 0.0
    total_tokens = 0
    grad_norms = []
    for inputs, targets, _ in loader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        optimizer.zero_grad()
        logits, _ = model(inputs)
        loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), clip_value)
        optimizer.step()
        non_pad = (targets != criterion.ignore_index).sum().item()
        total_loss += loss.item() * non_pad
        total_tokens += non_pad
        grad_norms.append(float(grad_norm))
    avg_loss = total_loss / max(total_tokens, 1)
    return avg_loss, perplexity_from_loss(avg_loss), grad_norms


@torch.no_grad()
def evaluate_language_model(model, loader, pad_idx):
    criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    for inputs, targets, _ in loader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        logits, _ = model(inputs)
        loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        non_pad = (targets != pad_idx).sum().item()
        total_loss += loss.item() * non_pad
        total_tokens += non_pad
    avg_loss = total_loss / max(total_tokens, 1)
    return {"loss": avg_loss, "perplexity": perplexity_from_loss(avg_loss)}


def train_language_model(model, train_loader, val_loader, clip_value):
    criterion = nn.CrossEntropyLoss(ignore_index=tgt_vocab.pad_idx)
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    history = {
        "train_loss": [],
        "val_loss": [],
        "train_accuracy": [],
        "val_accuracy": [],
        "grad_norm": [],
    }
    best_state = deepcopy(model.state_dict())
    best_val_loss = float("inf")
    patience = 0

    for _ in range(CONFIG["lm_epochs"]):
        train_loss, train_ppl, grad_norms = lm_train_epoch(
            model, train_loader, optimizer, criterion, clip_value
        )
        val_metrics = evaluate_language_model(model, val_loader, tgt_vocab.pad_idx)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["train_accuracy"].append(train_ppl)
        history["val_accuracy"].append(val_metrics["perplexity"])
        history["grad_norm"].append(sum(grad_norms) / max(len(grad_norms), 1))

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_state = deepcopy(model.state_dict())
            patience = 0
        else:
            patience += 1
        if patience >= CONFIG["patience"]:
            break

    model.load_state_dict(best_state)
    return model, history

In [ ]:
lm_records = []
for cell_type in ["rnn", "lstm", "gru"]:
    print(f"\n>>> Modèle de langage : {cell_type.upper()}")
    model = RecurrentLanguageModel(
        vocab_size=len(tgt_vocab),
        embedding_dim=CONFIG["embedding_dim"],
        hidden_dim=CONFIG["hidden_dim"],
        cell_type=cell_type,
        dropout=CONFIG["dropout"],
        pad_idx=tgt_vocab.pad_idx,
    ).to(device)

    start = time.perf_counter()
    model, history = train_language_model(model, train_lm_loader, val_lm_loader, CONFIG["clip_value"])
    elapsed = round(time.perf_counter() - start, 2)
    test_metrics = evaluate_language_model(model, test_lm_loader, tgt_vocab.pad_idx)

    lm_records.append(
        {
            "modele": cell_type.upper(),
            "loss_test": test_metrics["loss"],
            "perplexite_test": test_metrics["perplexity"],
            "temps_entrainement_s": elapsed,
        }
    )

    fig = plot_training_curves(history, f"LM {cell_type.upper()}")
    fig.savefig(FIGURES_DIR / f"lm_{cell_type}_curves.png", dpi=200, bbox_inches="tight")
    plt.show()

lm_df = pd.DataFrame(lm_records).sort_values("perplexite_test")
lm_df.to_csv(TABLES_DIR / "sequence_language_models_comparison.csv", index=False)
lm_df.to_latex(TABLES_DIR / "sequence_language_models_comparison.tex", index=False, float_format="%.4f")
display(lm_df.style.format({"loss_test": "{:.4f}", "perplexite_test": "{:.2f}"}))

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(lm_df["modele"], lm_df["perplexite_test"], color="#2ca02c")
ax.set_title("Comparaison des perplexités")
ax.set_ylabel("Perplexité")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "sequence_language_model_perplexity.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. Gradient clipping

In [ ]:
clipping_records = []
clipping_histories = {}

for experiment_name, clip_value in [
    ("avec_clipping", CONFIG["clip_value"]),
    ("sans_clipping", CONFIG["no_clip_value"]),
]:
    print(f"\n>>> Clipping : {experiment_name}")
    model = RecurrentLanguageModel(
        vocab_size=len(tgt_vocab),
        embedding_dim=CONFIG["embedding_dim"],
        hidden_dim=CONFIG["hidden_dim"],
        cell_type="gru",
        dropout=CONFIG["dropout"],
        pad_idx=tgt_vocab.pad_idx,
    ).to(device)
    model, history = train_language_model(model, train_lm_loader, val_lm_loader, clip_value)
    metrics = evaluate_language_model(model, test_lm_loader, tgt_vocab.pad_idx)
    clipping_histories[experiment_name] = history
    clipping_records.append(
        {
            "experience": experiment_name,
            "clip_value": str(clip_value),
            "perplexite_test": metrics["perplexity"],
            "norme_gradient_moyenne": sum(history["grad_norm"]) / max(len(history["grad_norm"]), 1),
        }
    )

clipping_df = pd.DataFrame(clipping_records)
clipping_df.to_csv(TABLES_DIR / "sequence_gradient_clipping.csv", index=False)
clipping_df.to_latex(TABLES_DIR / "sequence_gradient_clipping.tex", index=False, float_format="%.4f")
display(clipping_df)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(clipping_df["experience"], clipping_df["norme_gradient_moyenne"], color="#9467bd")
ax.set_title("Norme moyenne des gradients")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "sequence_gradient_norms.png", dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
for name, history in clipping_histories.items():
    ax.plot(history["val_loss"], label=name, linewidth=2)
ax.set_title("Perte validation — avec vs sans clipping")
ax.set_xlabel("Époque")
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / "sequence_clipping_comparison_curves.png", dpi=200, bbox_inches="tight")
plt.show()

## 5. Seq2Seq — entraînement, BLEU et exemples de traduction

In [ ]:
def seq2seq_train_epoch(model, loader, optimizer, criterion, teacher_forcing_ratio, clip_value):
    model.train()
    total_loss = 0.0
    total_tokens = 0
    for src_batch, tgt_batch, _, _ in loader:
        src_batch = src_batch.to(device)
        tgt_batch = tgt_batch.to(device)
        optimizer.zero_grad()
        outputs = model(src_batch, tgt_batch, teacher_forcing_ratio=teacher_forcing_ratio)
        loss = criterion(outputs.reshape(-1, outputs.size(-1)), tgt_batch[:, 1:].reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_value)
        optimizer.step()
        non_pad = (tgt_batch[:, 1:] != criterion.ignore_index).sum().item()
        total_loss += loss.item() * non_pad
        total_tokens += non_pad
    return total_loss / max(total_tokens, 1)


@torch.no_grad()
def seq2seq_eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    for src_batch, tgt_batch, _, _ in loader:
        src_batch = src_batch.to(device)
        tgt_batch = tgt_batch.to(device)
        outputs = model(src_batch, tgt_batch, teacher_forcing_ratio=0.0)
        loss = criterion(outputs.reshape(-1, outputs.size(-1)), tgt_batch[:, 1:].reshape(-1))
        non_pad = (tgt_batch[:, 1:] != criterion.ignore_index).sum().item()
        total_loss += loss.item() * non_pad
        total_tokens += non_pad
    return total_loss / max(total_tokens, 1)


def train_seq2seq(model):
    criterion = nn.CrossEntropyLoss(ignore_index=tgt_vocab.pad_idx)
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    history = {key: [] for key in ["train_loss", "val_loss", "train_accuracy", "val_accuracy"]}
    best_state = deepcopy(model.state_dict())
    best_val_loss = float("inf")
    patience = 0

    for _ in range(CONFIG["seq2seq_epochs"]):
        train_loss = seq2seq_train_epoch(
            model,
            train_translation_loader,
            optimizer,
            criterion,
            CONFIG["teacher_forcing_ratio"],
            CONFIG["clip_value"],
        )
        val_loss = seq2seq_eval_epoch(model, val_translation_loader, criterion)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_accuracy"].append(perplexity_from_loss(train_loss))
        history["val_accuracy"].append(perplexity_from_loss(val_loss))

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = deepcopy(model.state_dict())
            patience = 0
        else:
            patience += 1
        if patience >= CONFIG["patience"]:
            break

    model.load_state_dict(best_state)
    return model, history


@torch.no_grad()
def greedy_decode(model, src_tensor, sos_idx, eos_idx, max_len):
    model.eval()
    _, hidden = model.encoder(src_tensor)
    input_token = torch.tensor([sos_idx], device=src_tensor.device)
    generated = []
    for _ in range(max_len):
        logits, hidden = model.decoder(input_token, hidden)
        next_token = logits.argmax(dim=1)
        token_id = int(next_token.item())
        if token_id == eos_idx:
            break
        generated.append(token_id)
        input_token = next_token
    return generated


@torch.no_grad()
def beam_search_decode(model, src_tensor, sos_idx, eos_idx, max_len, beam_width=3):
    model.eval()
    _, initial_hidden = model.encoder(src_tensor)
    beams = [([], 0.0, initial_hidden, torch.tensor([sos_idx], device=src_tensor.device), False)]

    for _ in range(max_len):
        candidates = []
        for sequence, score, hidden, input_token, finished in beams:
            if finished:
                candidates.append((sequence, score, hidden, input_token, finished))
                continue
            logits, next_hidden = model.decoder(input_token, hidden)
            log_probs = torch.log_softmax(logits, dim=1)
            top_scores, top_indices = torch.topk(log_probs, beam_width, dim=1)
            for beam_score, token_idx in zip(top_scores[0], top_indices[0]):
                token_id = int(token_idx.item())
                next_sequence = sequence + ([] if token_id == eos_idx else [token_id])
                candidates.append(
                    (
                        next_sequence,
                        score + float(beam_score.item()),
                        next_hidden,
                        token_idx.unsqueeze(0),
                        token_id == eos_idx,
                    )
                )
        beams = sorted(
            candidates,
            key=lambda item: item[1] / math.sqrt(len(item[0]) + 1),
            reverse=True,
        )[:beam_width]
        if all(finished for _, _, _, _, finished in beams):
            break
    return beams[0][0]


def corpus_bleu_score(references, hypotheses):
    smoothing = SmoothingFunction().method1
    return float(corpus_bleu(references, hypotheses, smoothing_function=smoothing))


@torch.no_grad()
def evaluate_seq2seq(model, loader, max_len, beam_width):
    references = []
    greedy_hypotheses = []
    beam_hypotheses = []
    examples = []

    for src_batch, tgt_batch, _, _ in loader:
        src_batch = src_batch.to(device)
        for idx in range(src_batch.size(0)):
            src_tensor = src_batch[idx : idx + 1]
            tgt_tensor = tgt_batch[idx]
            reference_tokens = tgt_vocab.decode(tgt_tensor.tolist())
            greedy_tokens = tgt_vocab.decode(
                greedy_decode(model, src_tensor, tgt_vocab.sos_idx, tgt_vocab.eos_idx, max_len)
            )
            beam_tokens = tgt_vocab.decode(
                beam_search_decode(
                    model,
                    src_tensor,
                    tgt_vocab.sos_idx,
                    tgt_vocab.eos_idx,
                    max_len,
                    beam_width,
                )
            )
            references.append([reference_tokens])
            greedy_hypotheses.append(greedy_tokens)
            beam_hypotheses.append(beam_tokens)
            if len(examples) < 8:
                examples.append(
                    {
                        "source": " ".join(src_vocab.decode(src_tensor.squeeze(0).tolist())),
                        "reference": " ".join(reference_tokens),
                        "greedy": " ".join(greedy_tokens),
                        "beam": " ".join(beam_tokens),
                    }
                )

    return {
        "bleu_greedy": corpus_bleu_score(references, greedy_hypotheses),
        "bleu_beam": corpus_bleu_score(references, beam_hypotheses),
        "examples_df": pd.DataFrame(examples),
    }

In [ ]:
encoder = Encoder(len(src_vocab), CONFIG["embedding_dim"], CONFIG["hidden_dim"], src_vocab.pad_idx, CONFIG["dropout"])
decoder = Decoder(len(tgt_vocab), CONFIG["embedding_dim"], CONFIG["hidden_dim"], tgt_vocab.pad_idx, CONFIG["dropout"])
seq2seq_model = Seq2Seq(encoder, decoder, tgt_vocab.pad_idx).to(device)

print("Entraînement Seq2Seq...")
seq2seq_model, seq2seq_history = train_seq2seq(seq2seq_model)
fig = plot_training_curves(seq2seq_history, "Seq2Seq GRU")
fig.savefig(FIGURES_DIR / "seq2seq_curves.png", dpi=200, bbox_inches="tight")
plt.show()

translation_eval = evaluate_seq2seq(
    seq2seq_model,
    test_translation_loader,
    CONFIG["max_decode_len"],
    CONFIG["beam_width"],
)

bleu_df = pd.DataFrame(
    [
        {"decodeur": "Greedy", "bleu": translation_eval["bleu_greedy"]},
        {"decodeur": "Beam search", "bleu": translation_eval["bleu_beam"]},
    ]
)
bleu_df.to_csv(TABLES_DIR / "sequence_seq2seq_bleu.csv", index=False)
bleu_df.to_latex(TABLES_DIR / "sequence_seq2seq_bleu.tex", index=False, float_format="%.4f")
translation_eval["examples_df"].to_csv(TABLES_DIR / "sequence_translation_examples.csv", index=False)
translation_eval["examples_df"].to_latex(
    TABLES_DIR / "sequence_translation_examples.tex", index=False, float_format="%.4f"
)

display(bleu_df.style.format({"bleu": "{:.4f}"}))
display(translation_eval["examples_df"])

torch.save(
    {
        "state_dict": seq2seq_model.state_dict(),
        "src_vocab": src_vocab.itos,
        "tgt_vocab": tgt_vocab.itos,
    },
    MODELS_DIR / "best_seq2seq_model.pt",
)
print("Modèle Seq2Seq sauvegardé.")